## Step 1: Install and import libraries

In [1]:
# Run this if required in your Jupyter environment:
# !pip install pandas numpy scikit-learn streamlit plotly pydeck joblib

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, classification_report


## Step 2: Load the dataset

In [2]:
DATA_PATH = r"C:\\Users\\HP\\AQi-Project\\INDIA_AQI_COMPLETE_20251126.csv"

use_columns = [
    "City", "State", "Latitude", "Longitude", "Datetime", "Month", "Day", "Hour", "Is_Weekend",
    "Season", "Time_of_Day", "Temp_2m_C", "Humidity_Percent", "Humidity_Category",
    "Wind_Speed_10m_kmh", "Wind_Category", "Precipitation_mm", "Pressure_MSL_hPa",
    "Cloud_Cover_Percent", "PM2_5_ugm3", "PM10_ugm3", "CO_ugm3", "NO2_ugm3",
    "SO2_ugm3", "O3_ugm3", "AOD", "US_AQI"
]

df = pd.read_csv(DATA_PATH, usecols=use_columns)
df["Datetime"] = pd.to_datetime(df["Datetime"], dayfirst=True, errors="coerce")
df.head()


,City,State,Latitude,Longitude,Datetime,Month,Day,Hour,Is_Weekend,Season,...,Pressure_MSL_hPa,Cloud_Cover_Percent,PM2_5_ugm3,PM10_ugm3,CO_ugm3,NO2_ugm3,SO2_ugm3,O3_ugm3,AOD,US_AQI
0,Agartala,Tripura,23.8315,91.2868,2022-08-05 00:00:00,8,5,0,0,Monsoon,...,1004.4,100,14.8,21.5,197,21.8,2.7,32,0.14,NaN
1,Agartala,Tripura,23.8315,91.2868,2022-08-05 01:00:00,8,5,1,0,Monsoon,...,1003.9,100,15.7,22.8,196,18.5,3.0,33,0.14,NaN
2,Agartala,Tripura,23.8315,91.2868,2022-08-05 02:00:00,8,5,2,0,Monsoon,...,1003.5,100,16.3,23.5,196,15.1,3.3,34,0.15,NaN
3,Agartala,Tripura,23.8315,91.2868,2022-08-05 03:00:00,8,5,3,0,Monsoon,...,1003.0,100,17.6,25.4,197,14.1,3.3,32,0.15,NaN
4,Agartala,Tripura,23.8315,91.2868,2022-08-05 04:00:00,8,5,4,0,Monsoon,...,1002.9,100,18.2,26.2,199,13.9,3.2,30,0.14,NaN


## Step 3: Basic dataset information

In [3]:
print("Shape:", df.shape)
print("Cities:", df["City"].nunique())
print("States:", df["State"].nunique())
df.describe()


Shape: (842160, 27)
Cities: 29
States: 29


,Latitude,Longitude,Datetime,Month,Day,Hour,Is_Weekend,Temp_2m_C,Humidity_Percent,Wind_Speed_10m_kmh,...,Pressure_MSL_hPa,Cloud_Cover_Percent,PM2_5_ugm3,PM10_ugm3,CO_ugm3,NO2_ugm3,SO2_ugm3,O3_ugm3,AOD,US_AQI
count,842160.000000,842160.000000,842160,842160.000000,842160.000000,842160.000000,842160.000000,842160.000000,842160.000000,842160.000000,...,842160.000000,842160.00000,842160.000000,842160.000000,842160.000000,842160.000000,842160.000000,842160.000000,842160.000000,842015.000000
mean,23.129041,82.769207,2024-03-31 23:30:00,6.801653,15.736364,11.500000,0.285950,23.628438,71.324796,8.117679,...,1009.489776,50.50123,34.739642,56.257083,450.891150,16.132833,13.599054,80.479350,0.465470,96.804419
min,8.524100,72.571400,2022-08-05 00:00:00,1.000000,1.000000,0.000000,0.000000,-11.500000,3.000000,0.000000,...,979.400000,0.00000,0.100000,0.100000,56.000000,-0.200000,-0.200000,-11.000000,0.000000,9.000000
25%,20.296100,77.173400,2023-06-03 11:45:00,4.000000,8.000000,5.750000,0.000000,19.600000,58.000000,4.300000,...,1005.400000,2.00000,14.700000,21.500000,235.000000,3.700000,2.800000,47.000000,0.240000,59.000000
50%,23.831500,80.946200,2024-03-31 23:30:00,7.000000,16.000000,11.500000,0.000000,24.700000,76.000000,7.100000,...,1009.900000,47.00000,25.900000,39.000000,333.000000,9.000000,6.900000,73.000000,0.400000,84.000000
75%,26.912400,88.606500,2025-01-28 11:15:00,10.000000,23.000000,17.250000,1.000000,28.100000,89.000000,10.800000,...,1013.800000,100.00000,44.200000,66.400000,508.000000,19.900000,15.800000,108.000000,0.610000,128.000000
max,31.104800,94.108600,2025-11-26 23:00:00,12.000000,31.000000,23.000000,1.000000,46.400000,100.000000,53.100000,...,1028.100000,100.00000,581.100000,3263.400000,14504.000000,336.000000,424.600000,930.000000,4.340000,2742.000000
std,5.543478,6.937205,NaN,3.411504,8.767003,6.922191,0.451866,6.861838,21.362100,5.077763,...,5.698795,43.41348,30.843496,83.837033,434.552149,20.813405,20.617020,46.196459,0.308188,70.784455


## Step 4: Clean missing values and create AQI category labels

In [4]:
def aqi_category(value):
    if pd.isna(value):
        return np.nan
    if value <= 50:
        return "Good"
    if value <= 100:
        return "Satisfactory"
    if value <= 200:
        return "Moderate"
    if value <= 300:
        return "Poor"
    if value <= 400:
        return "Very Poor"
    return "Severe"

def compact_category(value):
    if pd.isna(value):
        return np.nan
    if value <= 50:
        return "Good"
    if value <= 200:
        return "Moderate"
    return "Poor"

df = df.dropna(subset=["US_AQI", "Latitude", "Longitude", "City", "State"])
df["AQI_Level"] = df["US_AQI"].apply(aqi_category)
df["Pollution_Level"] = df["US_AQI"].apply(compact_category)

numeric_features = [
    "Latitude", "Longitude", "Month", "Day", "Hour", "Is_Weekend", "Temp_2m_C",
    "Humidity_Percent", "Wind_Speed_10m_kmh", "Precipitation_mm", "Pressure_MSL_hPa",
    "Cloud_Cover_Percent", "PM2_5_ugm3", "PM10_ugm3", "CO_ugm3", "NO2_ugm3",
    "SO2_ugm3", "O3_ugm3", "AOD"
]
categorical_features = ["City", "State", "Season", "Time_of_Day", "Humidity_Category", "Wind_Category"]

for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

for col in categorical_features:
    df[col] = df[col].fillna("Unknown").astype(str)

df[["City", "US_AQI", "AQI_Level", "Pollution_Level"]].head()


,City,US_AQI,AQI_Level,Pollution_Level
5,Agartala,54.0,Satisfactory,Moderate
6,Agartala,55.0,Satisfactory,Moderate
7,Agartala,55.0,Satisfactory,Moderate
8,Agartala,55.0,Satisfactory,Moderate
9,Agartala,54.0,Satisfactory,Moderate


## Step 5: Prepare train-test data

In [5]:
# Use a sample for faster training in college/laptop environments.
sample_df = df.sample(50000, random_state=42) if len(df) > 50000 else df.copy()

X = sample_df[numeric_features + categorical_features]
y_regression = sample_df["US_AQI"]
y_classification = sample_df["Pollution_Level"]

def build_preprocessor():
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    return ColumnTransformer([
        ("numeric", StandardScaler(), numeric_features),
        ("category", encoder, categorical_features),
    ])

X_train, X_test, y_reg_train, y_reg_test = train_test_split(X, y_regression, test_size=0.2, random_state=42)
X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(X, y_classification, test_size=0.2, random_state=42, stratify=y_classification)


## Step 6: Regression models for AQI prediction

In [6]:
regression_models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(max_depth=12, random_state=42),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=80, max_depth=14, random_state=42, n_jobs=-1),
}

trained_regression = {}
regression_results = []

for name, model in regression_models.items():
    pipe = Pipeline([("preprocess", build_preprocessor()), ("model", model)])
    pipe.fit(X_train, y_reg_train)
    pred = pipe.predict(X_test)
    trained_regression[name] = pipe
    regression_results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_reg_test, pred),
        "R2 Score": r2_score(y_reg_test, pred),
    })

pd.DataFrame(regression_results)


,Model,MAE,R2 Score
0,Linear Regression,17.391391,0.741304
1,Decision Tree Regressor,15.673633,0.672922
2,Random Forest Regressor,13.265715,0.801089


## Step 7: Classification models for pollution level prediction

In [7]:
classification_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree Classifier": DecisionTreeClassifier(max_depth=12, random_state=42),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=80, max_depth=14, random_state=42, n_jobs=-1),
}

trained_classification = {}
classification_results = []

for name, model in classification_models.items():
    pipe = Pipeline([("preprocess", build_preprocessor()), ("model", model)])
    pipe.fit(X_cls_train, y_cls_train)
    pred = pipe.predict(X_cls_test)
    trained_classification[name] = pipe
    classification_results.append({"Model": name, "Accuracy": accuracy_score(y_cls_test, pred)})
    print("\n", name)
    print(classification_report(y_cls_test, pred))

pd.DataFrame(classification_results)



 Logistic Regression
              precision    recall  f1-score   support

        Good       0.81      0.81      0.81      1615
    Moderate       0.95      0.96      0.96      8218
        Poor       0.78      0.41      0.54       167

    accuracy                           0.93     10000
   macro avg       0.85      0.73      0.77     10000
weighted avg       0.93      0.93      0.93     10000


 Decision Tree Classifier
              precision    recall  f1-score   support

        Good       0.77      0.76      0.76      1615
    Moderate       0.94      0.95      0.95      8218
        Poor       0.66      0.59      0.62       167

    accuracy                           0.91     10000
   macro avg       0.79      0.76      0.78     10000
weighted avg       0.91      0.91      0.91     10000


 Random Forest Classifier
              precision    recall  f1-score   support

        Good       0.85      0.79      0.82      1615
    Moderate       0.95      0.97      0.96      8218

,Model,Accuracy
0,Logistic Regression,0.9275
1,Decision Tree Classifier,0.9124
2,Random Forest Classifier,0.9309


## Step 8: K-Means clustering for pollution patterns

In [8]:
cluster_features = ["PM2_5_ugm3", "PM10_ugm3", "CO_ugm3", "NO2_ugm3", "SO2_ugm3", "O3_ugm3", "US_AQI"]
cluster_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("kmeans", KMeans(n_clusters=4, random_state=42, n_init=10)),
])

sample_df["Cluster"] = cluster_pipeline.fit_predict(sample_df[cluster_features])
sample_df.groupby("Cluster")[["US_AQI", "PM2_5_ugm3", "PM10_ugm3", "NO2_ugm3"]].mean().round(2)


,US_AQI,PM2_5_ugm3,PM10_ugm3,NO2_ugm3
Cluster,,,,
0,166.82,104.70,155.29,73.94
1,65.52,18.48,27.94,9.49
2,944.41,143.92,1087.74,19.58
3,136.59,51.57,81.99,17.22


## Step 9: City AQI data for India map

In [9]:
city_aqi = (
    df.groupby(["City", "State", "Latitude", "Longitude"], as_index=False)
      .agg(Average_AQI=("US_AQI", "mean"), Records=("US_AQI", "size"))
)
city_aqi["AQI_Level"] = city_aqi["Average_AQI"].apply(aqi_category)
city_aqi.sort_values("Average_AQI", ascending=False).head(10)


,City,State,Latitude,Longitude,Average_AQI,Records,AQI_Level
11,Gurugram,Haryana,28.4595,77.0266,222.521681,29035,Poor
9,Delhi,Delhi,28.6139,77.2090,159.843637,29035,Moderate
22,Patna,Bihar,25.5941,85.1376,132.851076,29035,Moderate
19,Lucknow,Uttar Pradesh,26.8467,80.9462,129.405579,29035,Moderate
23,Raipur,Chhattisgarh,21.2514,81.6296,123.694162,29035,Moderate
18,Kolkata,West Bengal,22.5726,88.3639,122.459755,29035,Moderate
6,Chandigarh,Punjab,30.7333,76.7794,115.338006,29035,Moderate
20,Mumbai,Maharashtra,19.0760,72.8777,114.228999,29035,Moderate
0,Agartala,Tripura,23.8315,91.2868,105.529292,29035,Moderate
5,Bhubaneswar,Odisha,20.2961,85.8245,103.731083,29035,Moderate


## Step 10: Save models

In [10]:
joblib.dump(trained_regression, "regression_models.pkl")
joblib.dump(trained_classification, "classification_models.pkl")
joblib.dump(cluster_pipeline, "kmeans_model.pkl")
print("Models saved successfully.")


Models saved successfully.


## Step 11: Run Streamlit UI

Open a terminal in this project folder and run:

```powershell
streamlit run app.py
```

In [1]:
import os
print("your project folder is:")
print(os.getcwd())
print("\nALL files:")
for f in os.listdir():
   print(f)


your project folder is:
C:\Users\HP

ALL files:
.anaconda
.bash_history
.blackbox-editor
.cache
.codex
.conda
.config
.continuum
.copilot
.cursor
.dbclient
.gitconfig
.idlerc
.ipynb_checkpoints
.ipython
.jupyter
.m2
.matplotlib
.ms-ad
.node_repl_history
.openjfx
.packettracer
.python_history
.redhat
.rsp
.streamlit
.VirtualBox
.vscode
.vscode-shared
aiml_project
anaconda3
app.py
AppData
Application Data
AQI-Prediction.ipynb
AQi-Project
AQI_Project_Notebook.ipynb
breast_cancer.csv
Cisco Packet Tracer 9.0.0
classification_models.pkl
Contacts
Cookies
CrossDevice
Desktop
Documents
Downloads
Dropbox
Favorites
final_aqi_dataset.csv
kmeans_model.pkl
Links
Local Settings
logs
Microsoft
Music
My Documents
naivebayes.ipynb
NetHood
NTUSER.DAT
ntuser.dat.LOG1
ntuser.dat.LOG2
NTUSER.DAT{fbdb099f-cbd3-11ef-9686-84ea9252c9fb}.TM.blf
NTUSER.DAT{fbdb099f-cbd3-11ef-9686-84ea9252c9fb}.TMContainer00000000000000000001.regtrans-ms
NTUSER.DAT{fbdb099f-cbd3-11ef-9686-84ea9252c9fb}.TMContainer00000000000000000

In [2]:
import os
import shutil

# Create a new clean folder
project_folder = r"C:\Users\AQI-Project-AiMl"
os.makedirs(project_folder, exist_ok=True)

# List of files to copy
files_to_copy = [
    "app.py",
    "INDIA_AQI_COMPLETE_20251126.csv",
    "regression_models.pkl",
    "classification_models.pkl",
    "kmeans_model.pkl",
    "requirements.txt"
]

# Copy each file to the new folder
for file in files_to_copy:
    src = r"C:\Users\\" + file
    dst = project_folder + "\\" + file
    shutil.copy(src, dst)
    print(f"✅ Copied: {file}")

print("\n✅ All files copied to:", project_folder)

PermissionError: [WinError 5] Access is denied: 'C:\\Users\\AQI-Project-AiMl'